<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. The rule in plain words

A page is worth a refresh look if it still gets search traffic and at least one of these is true:

1. **Stale but visible** — not updated in 180+ days, and still has at least 500 impressions.
2. **Position risk** — older than 180 days and sitting past position 10.
3. **Low engagement** — at least 1,000 impressions but CTR under 0.05 (that is 0.05%, because `ctr` is stored ×100).

Score = those flags × impressions, with fixed weights 0.4 / 0.3 / 0.3. No fitted weights. No `trend_direction`. No `trend_pct`. Reason codes say which flags fired.

In [1]:
import sys
from pathlib import Path

repo_root = None
for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    scripts = p / "work" / "scripts"
    if (scripts / "warehouse_frame.py").exists():
        sys.path.insert(0, str(scripts))
        repo_root = p
        break
else:
    raise FileNotFoundError("work/scripts/warehouse_frame.py not found")

from warehouse_frame import load_notebook_frame

df = load_notebook_frame()
print(f"Outputs folder: {repo_root / 'work' / 'outputs'}")

Hugging Face warehouse: 79,576 pages, 26 clients
Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.
Declining rate: 0.557
Outputs folder: C:\Users\rimla\Downloads\flyrank-ml-assignments-main\flyrank-ml-assignments-main\work\outputs


In [2]:
print("Fair rule (knowable at decision time; no label in the score):")
print("  STALE_VISIBLE:        days_since_last_update >= 180 AND impressions_90d >= 500")
print("  POSITION_DECAY_RISK:  avg_position > 10 AND content_age_days >= 180")
print("  LOW_ENGAGEMENT:       ctr < 0.05 AND impressions_90d >= 1000")
print("score = 0.4*stale*impressions + 0.3*position*impressions + 0.3*low_ctr*impressions")

Fair rule (knowable at decision time; no label in the score):
  STALE_VISIBLE:        days_since_last_update >= 180 AND impressions_90d >= 500
  POSITION_DECAY_RISK:  avg_position > 10 AND content_age_days >= 180
  LOW_ENGAGEMENT:       ctr < 0.05 AND impressions_90d >= 1000
score = 0.4*stale*impressions + 0.3*position*impressions + 0.3*low_ctr*impressions


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The score is a weighted sum of three flags times impressions. Pages with score 0 are MONITOR. The ranked file goes to `work/outputs/baseline_action_score.csv` (queue columns only).


In [3]:
from pathlib import Path
import numpy as np

if "repo_root" not in locals():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "AGENTS.md").exists():
            repo_root = p
            break
    else:
        raise FileNotFoundError("repo root not found")

stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int)
position_decay_risk = ((df["avg_position"] > 10) & (df["content_age_days"] >= 180)).astype(int)
low_engagement = ((df["ctr"] < 0.05) & (df["impressions_90d"] >= 1000)).astype(int)

df["stale_visible_flag"] = stale_visible
df["position_decay_flag"] = position_decay_risk
df["low_engagement_flag"] = low_engagement

def assign_reason_code(row):
    reasons = []
    if row["stale_visible_flag"] == 1:
        reasons.append("STALE_VISIBLE")
    if row["position_decay_flag"] == 1:
        reasons.append("POSITION_DECAY_RISK")
    if row["low_engagement_flag"] == 1:
        reasons.append("LOW_ENGAGEMENT")
    return "_".join(reasons) if reasons else "MONITOR"

df["reason_code"] = df.apply(assign_reason_code, axis=1)
df["baseline_score"] = (
    0.4 * stale_visible * df["impressions_90d"]
    + 0.3 * position_decay_risk * df["impressions_90d"]
    + 0.3 * low_engagement * df["impressions_90d"]
)
df["action_label"] = np.where(df["baseline_score"] > 0, "CONTENT_REFRESH_PRIORITY", "MONITOR")

ranked_queue = df.sort_values(by="baseline_score", ascending=False)

out_dir = repo_root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
queue_cols = [
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
]
out_path = out_dir / "baseline_action_score.csv"
ranked_queue[queue_cols].to_csv(out_path, index=False)
print(f"Wrote {len(ranked_queue):,} rows to {out_path}")
print(f"Flagged for refresh: {(df['action_label'] == 'CONTENT_REFRESH_PRIORITY').sum():,}")
print(f"Monitor: {(df['action_label'] == 'MONITOR').sum():,}")

Wrote 79,576 rows to C:\Users\rimla\Downloads\flyrank-ml-assignments-main\flyrank-ml-assignments-main\work\outputs\baseline_action_score.csv
Flagged for refresh: 23,986
Monitor: 55,590


Precision@K is computed next. Full-catalog ranking is shown for context; the number used later is **client-holdout fold 1**, same split as the model.

In [4]:
import numpy as np
from sklearn.model_selection import GroupKFold

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = (df["trend_direction"].str.lower() == "down").astype(int)

print("=== FULL CATALOG (not the reported number) ===")
for k in [20, 50, 100]:
    p = precision_at_k(df["baseline_score"], y, k)
    print(f"Precision@{k}: {p:.3f} ({int(round(p * k))}/{k})")
print(f"Base rate: {y.mean():.3f}")

print("\n=== CLIENT-HOLDOUT FOLD 1 (reported; same split as the model) ===")
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(df, y, groups=df["client_id"]))
p50 = precision_at_k(df.iloc[test_idx]["baseline_score"], y.iloc[test_idx], 50)
print(f"Train: {len(train_idx):,}  Test: {len(test_idx):,}")
print(f"Fair baseline Precision@50: {p50:.3f} ({int(round(p50 * 50))}/50)")
print(f"Test base rate: {y.iloc[test_idx].mean():.3f}")
print(f"vs random on this fold: {p50 / y.iloc[test_idx].mean():.2f}x")

=== FULL CATALOG (not the reported number) ===
Precision@20: 0.650 (13/20)
Precision@50: 0.500 (25/50)
Precision@100: 0.530 (53/100)
Base rate: 0.557

=== CLIENT-HOLDOUT FOLD 1 (reported; same split as the model) ===
Train: 58,783  Test: 20,793
Fair baseline Precision@50: 0.640 (32/50)
Test base rate: 0.439
vs random on this fold: 1.46x


The reported bar is that fold-1 Precision@50 (0.640 in `canonical_metrics.json`). Full-catalog ranking is usually a bit different because it is not the same holdout fold. Weak picks and the leak check are in section 4.

In [5]:
print("Holdout Precision@50 is in the cell above. Top-20 review is next.")

Holdout Precision@50 is in the cell above. Top-20 review is next.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
print("=== TOP-20 BASELINE REVIEW ===")
top_20 = ranked_queue.head(20)
for rank, (idx, row) in enumerate(top_20.iterrows(), start=1):
    print(f"\n#{rank}  action={row['action_label']}  reason={row['reason_code']}")
    print(
        f"  impressions={row['impressions_90d']:,.0f}  position={row['avg_position']:.1f}  "
        f"age={row['content_age_days']:.0f}d  ctr={row['ctr']:.3f}  "
        f"days_since_update={row['days_since_last_update']:.0f}"
    )
    notes = []
    if "STALE_VISIBLE" in row["reason_code"]:
        notes.append("wrong if evergreen and age is fine")
    if "POSITION_DECAY_RISK" in row["reason_code"]:
        notes.append("wrong if position is stable on purpose")
    if "LOW_ENGAGEMENT" in row["reason_code"]:
        notes.append("wrong if this CTR is normal for the page type")
    if row["days_since_last_update"] < 30:
        notes.append("WEAK: updated in the last month")
    if row["avg_position"] > 0 and row["avg_position"] <= 3:
        notes.append("WEAK: already in the top 3")
    print("  " + "; ".join(notes) if notes else "  look by hand before editing")


=== TOP-20 BASELINE REVIEW ===

#1  action=CONTENT_REFRESH_PRIORITY  reason=POSITION_DECAY_RISK
  impressions=263,208  position=12.4  age=200d  ctr=0.422  days_since_update=-102
  wrong if position is stable on purpose; WEAK: updated in the last month

#2  action=CONTENT_REFRESH_PRIORITY  reason=LOW_ENGAGEMENT
  impressions=227,674  position=0.3  age=380d  ctr=0.001  days_since_update=-113
  wrong if this CTR is normal for the page type; WEAK: updated in the last month; WEAK: already in the top 3

#3  action=CONTENT_REFRESH_PRIORITY  reason=LOW_ENGAGEMENT
  impressions=208,576  position=0.2  age=380d  ctr=0.000  days_since_update=-113
  wrong if this CTR is normal for the page type; WEAK: updated in the last month; WEAK: already in the top 3

#4  action=CONTENT_REFRESH_PRIORITY  reason=LOW_ENGAGEMENT
  impressions=195,713  position=0.0  age=213d  ctr=0.001  days_since_update=4
  wrong if this CTR is normal for the page type; WEAK: updated in the last month; WEAK: already in the top 3



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

High-impression pages dominate the top of the list because the score multiplies by `impressions_90d`. That is readable, but it also means:

- A page already in the top 3 with low CTR can still rank high (`LOW_ENGAGEMENT` only).
- A page updated in the last month can still rank high on `POSITION_DECAY_RISK` because that flag uses age, not freshness.
- Evergreen or seasonal pages can look "stale" without needing a rewrite.

Those are the cases a person should skip.

### Leakage

The score uses only age, freshness, impressions, position, and CTR. It does **not** use `trend_direction`, `trend_pct`, product flags, or any future window. The next cell checks that.


In [7]:
print("=== LEAKAGE CHECK ===")
banned = ["trend_pct", "trend_direction", "is_declining_label", "health_score", "priority_score"]
used = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "ctr",
]
print("Inputs in the score:", ", ".join(used))
print("Banned columns in the dataframe:", [c for c in banned if c in df.columns])
print("Banned columns used as score inputs: none")

y = (df["trend_direction"].str.lower() == "down").astype(int)
corr = df["baseline_score"].corr(y)
print(f"\nCorrelation of baseline_score with declining label: {corr:.3f}")
print("(A value near 1.0 would mean the score is reading the label. Mid/low is expected.)")

assert "trend_pct" not in used
assert "trend_direction" not in used
print("\nPass: no label-derived or product-flag inputs in the fair rule.")


=== LEAKAGE CHECK ===
Inputs in the score: days_since_last_update, impressions_90d, avg_position, content_age_days, ctr
Banned columns in the dataframe: ['trend_pct', 'trend_direction']
Banned columns used as score inputs: none

Correlation of baseline_score with declining label: 0.005
(A value near 1.0 would mean the score is reading the label. Mid/low is expected.)

Pass: no label-derived or product-flag inputs in the fair rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.